In [1]:
import os
import pandas as pd
import numpy as np
from numba import njit, float64, int64, types
from numba.typed import Dict

In [2]:

# 현재 파일들이 있는 그 위치 그대로 설정
Base_dir = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"

# 파일 이름에 포함된 단어로 공격 유형 구분
attack_mapping = {
    "Dos": 1,
    "Fuzzing": 2,
    "Spoofing":4
}

attack_files = []

# 폴더 안을 바로 검사
for attack_name, attack_id in attack_mapping.items():
    attack_dir = os.path.join(Base_dir, attack_name)
    if not os.path.isdir(attack_dir):
        continue

    for fname in os.listdir(attack_dir):
        if fname.endswith(".csv"):
            attack_files.append({
                "path": os.path.join(attack_dir, fname),
                "attack_id": attack_id
            })

In [3]:
#################################
# 2. Visualization Mirgu Dataset
#################################
hash_cache = {}

# [ADD] payload 8바이트 리스트로 만드는 함수 (너가 쓰던 스타일)
def parse_payload(row):
    # row에는 b0~b7 컬럼이 있고, 이미 0패딩되어 있음
    return [int(row[f"b{i}"]) for i in range(8)]

def process_csv_file(path, attack_id):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) < 4:
                continue

            ts_str, canid_raw, dlc_str = parts[0], parts[1], parts[2]
            label = parts[-1].strip()         # [MINOR] strip
            data_tokens = parts[3:-1]

            # dlc/ts 파싱
            try:
                ts = float(ts_str)
                dlc = int(dlc_str)
            except:
                continue

            # payload bytes: DLC 만큼만 읽고, 8바이트로 0 패딩
            payload = []
            for i in range(min(dlc, len(data_tokens), 8)):
                tok = data_tokens[i].strip()
                if tok == "" or tok.lower() == "nan":
                    payload.append(0)
                else:
                    try:
                        payload.append(int(tok, 16))
                    except:
                        payload.append(0)

            payload += [0] * (8 - len(payload))
            payload = payload[:8]

            rows.append([ts, canid_raw, dlc, *payload, label])

    df = pd.DataFrame(
        rows,
        columns=["timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(8)] + ["Label"]
    )

    # CAN_ID int 변환
    df["int_CAN_ID"] = df["CAN_ID"].apply(lambda x: int(str(x).strip(), 16)).astype(np.int64)


    # Payloads 컬럼 추가 
    df["Payloads"] = df.apply(parse_payload, axis=1).tolist()

    # 라벨링
    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0}).fillna(0).astype(int)

    df = df[["timestamp","int_CAN_ID","Payloads","Labeling"]]

    return df


In [4]:
# %%
@njit
def popcount64(x):
    # x: uint8 -> 0~255
    c = 0
    v = int64(x)
    while v:
        v &= v - np.uint64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = np.uint64(0)
    for i in range(8):
        v |= np.uint64(row[i]) << (i * 8)
    return v

@njit(fastmath=True)
def calculate_features_numba(timestamps, can_ids, payloads):
    n = len(timestamps)
    # 9개 피처: IAT, IsZero, Ent, Complex, HamRate, Freq, Continuity, DiffEnt, WinIdEnt
    features = np.zeros((n, 9), dtype=np.float64)
    
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)
    last_id_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # EMA용 맵 (Hamming 변화량 추적)
    id_ham_ema = Dict.empty(key_type=types.int64, value_type=types.float64)
    alpha = 0.05  # 지수 이동 평균 가중치
    
    prev_global_time = timestamps[0]
    eps = 1e-9

    for i in range(n):
        # 64개 패킷마다 윈도우 빈도 초기화
        if (i % 64) == 0:
            last_id_map.clear()
            
        ts = timestamps[i]
        cid = can_ids[i]
        row = payloads[i] # row 정의를 상단으로 이동
        
        if np.isnan(ts): 
            ts = prev_global_time
        else: 
            prev_global_time = ts

        # --- [1] Hamming 및 상대적 변화량 계산 ---
        cur_bytes = pack_payload_u64(row)
        rel_change = 0.0
        
        if cid in last_payload_map:
            diff = cur_bytes ^ last_payload_map[cid]
            h_dist = float64(popcount64(diff))
            
            if cid in id_ham_ema:
                avg_h = id_ham_ema[cid]
                rel_change = h_dist / (avg_h + 0.1) 
                id_ham_ema[cid] = (1.0 - alpha) * avg_h + alpha * h_dist
            else:
                rel_change = 1.0
                id_ham_ema[cid] = h_dist
        else:
            rel_change = 0.0
            
        last_payload_map[cid] = cur_bytes

        # --- [2] 피처별 할당 ---
        
        # 0: ID IAT
        if cid in last_time_map: 
            id_iat = max(0.0, ts - last_time_map[cid])
        else: 
            id_iat = 0.0
        features[i, 0] = np.log1p(id_iat * 1000.0) / 7.0
        last_time_map[cid] = ts
        
        # 1: Is Zero ID (DoS)
        features[i, 1] = 1.0 if cid == 0 else 0.0

        # 2: Payload Entropy
        p_counts = np.zeros(256, dtype=np.int64)
        for b in row:
            p_counts[b] += 1
        ent = 0.0
        for c in p_counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)
        features[i, 2] = ent / 2.1

        # 3: Complexity (Entropy * Rel_Change)
        features[i, 3] = np.log1p(ent * rel_change)

        # 4: Hamming Rate
        features[i, 4] = np.log1p(rel_change / (id_iat + eps)) / 10.0
             
        # 5: Local Frequency
        cnt = last_id_map.get(cid, 0.0) + 1.0
        last_id_map[cid] = cnt
        features[i, 5] = cnt / 128.0

        # 6: Payload Continuity
        features[i, 6] = np.log1p(rel_change) / 5.0

        # 7: Byte-level Differential Entropy
        diffs = np.zeros(7, dtype=np.int64)
        for b_idx in range(7):
            diffs[b_idx] = (int64(row[b_idx+1]) - int64(row[b_idx])) % 256
        
        d_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
        for d in diffs:
            d_counts[d] = d_counts.get(d, 0.0) + 1.0
        
        d_ent = 0.0
        for dv in d_counts:
            p = d_counts[dv] / 7.0
            d_ent -= p * np.log(p + 1e-9)
        features[i, 7] = d_ent / 1.94

        # 8: Window-level ID Entropy
        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for j in range(i-127, i+1):
                wid = can_ids[j]
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            
            wi_ent = 0.0
            for k_id in win_id_counts:
                pk = win_id_counts[k_id] / 128.0
                wi_ent -= pk * np.log(pk + 1e-9)
            features[i, 8] = wi_ent / 4.85
        else:
            features[i, 8] = 0.0

    return features

In [5]:
# ==========================================
# 4. Making Feature with Numba
# ==========================================

def Make_feature(path, attack_id):

    df = process_csv_file(path, attack_id)

    # ======== to numpy ========== #
    timestamps = df["timestamp"].to_numpy(np.float32)
    can_ids = df["int_CAN_ID"].to_numpy(np.int64)
    payloads = np.array(df["Payloads"].tolist(), dtype=np.uint8)
    labels = df["Labeling"].to_numpy(np.int64)

    
    # ======== calculate feature ========== #
    feature9 = calculate_features_numba(timestamps, can_ids, payloads)
    print(feature9.shape)

    return feature9, labels

In [6]:
# ==========================================
# 5. Slide Window and Label
# ==========================================
def Sliding_Window_and_Labeling(feature, label, win_size=128, stride=32):
    windows = []
    labels = []
    n = feature.shape[0]
    for start in range(0, n-win_size+1 , stride):
        end = start + win_size
        windows.append(feature[start:end])
        labels.append(label[start:end])


    return (
        np.stack(windows, axis=0).astype(np.float32),
        np.stack(labels, axis=0).astype(np.int64)
    )

In [7]:
# ==========================================
# 6. main
# ==========================================
all_x = []
all_y = []

for item in attack_files:
    feature9, labels = Make_feature(item["path"], item["attack_id"]) # 각 feature 추출
    windows, y = Sliding_Window_and_Labeling(feature9,labels) # 윈도우 만들기

    all_x.append(windows)
    all_y.append(y)

all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

(3665771, 9)
(3838860, 9)
(4443142, 9)
(4621702, 9)


In [8]:
# ==========================================
# 7. Save
# ==========================================
import numpy as np
np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_0211_1208.npz",
    X = all_x_win.astype(np.float32),
    y = all_y_win.astype(np.int64)
    )

print(f" Saved dataset")

print("X shape:", all_x_win.shape)
print("y shape:", all_y_win.shape)

 Saved dataset
X shape: (517783, 128, 9)
y shape: (517783, 128)
